In [ ]:
# 1) Install dependencies (run once)
# --------------------------------------------------
# Vivarium core engine + public-health extras
!pip install vivarium-core vivarium-public-health      

# OpenFisca core + Australian rules from GitHub
!pip install openfisca-core                             # core engine :contentReference[oaicite:0]{index=0}
!pip install git+https://github.com/openfisca/openfisca-australia.git

  Using cached psutil-5.9.8-cp38-abi3-macosx_11_0_arm64.whl.metadata (21 kB)
Using cached psutil-5.9.8-cp38-abi3-macosx_11_0_arm64.whl (249 kB)
  Attempting uninstall: psutil
    Found existing installation: psutil 7.0.0
    Uninstalling psutil-7.0.0:
      Successfully uninstalled psutil-7.0.0
  Cloning https://github.com/openfisca/openfisca-australia.git to /private/var/folders/m9/_g3wndrn04d80ys2382r700r0000gn/T/pip-req-build-em36mqgo
  Running command git clone --filter=blob:none --quiet https://github.com/openfisca/openfisca-australia.git /private/var/folders/m9/_g3wndrn04d80ys2382r700r0000gn/T/pip-req-build-em36mqgo
Username for 'https://github.com': 

In [ ]:
# 2) Imports
# --------------------------------------------------
import numpy as np
import pandas as pd

# Vivarium interactive API
from vivarium import InteractiveContext  
from vivarium.framework.population import Population

# OpenFisca rule engine
from openfisca_core.simulation_builder import SimulationBuilder  
from openfisca_australia import CountryTaxBenefitSystem  

# 3) Define components
# --------------------------------------------------
class CostComponent:
    """Annual cost producer + PV cost accumulation."""
    CONFIGURATION_DEFAULTS = {
        'discount_rate': 0.035,
        'common': {
            'respite_care': 13024,
            'mental_health': 30418,
            'hospital': 6800,
            'informal_care': 25000,
        },
        'severity_specific': {
            'mild':     {'ndis_school': 26472,  'residential': 0},
            'moderate': {'ndis_school': 86472,  'residential': 0},
            'severe':   {'ndis_school':105972,  'residential':16973},
        },
        'dsp_annual': 23254,
    }

    def setup(self, builder):
        builder.value.register_value_producer('annual_cost', self.compute)
        builder.value.register_value_modifier('present_value_cost', self.accumulate)
        self.clock = builder.time.clock()

    def initialise_population(self, pop):
        pop['present_value_cost'] = 0.0
        pop['age_at_entry'] = pop['age']

    def compute(self, pop, _):
        cfg  = self.configuration
        base = sum(cfg.common.values())
        idx  = pop['severity'].cat.codes
        # vectorise severity-specific
        sev_keys = list(cfg.severity_specific.keys())
        ndis     = [cfg.severity_specific[sev_keys[i]]['ndis_school'] for i in idx]
        resi     = [cfg.severity_specific[sev_keys[i]]['residential'] for i in idx]
        dsp      = np.where(pop['age']>=20, cfg.dsp_annual, 0)
        return base + np.array(ndis) + np.array(resi) + dsp

    def accumulate(self, pop, annual, _):
        years = pop['age'] - pop['age_at_entry']
        r     = self.configuration.discount_rate
        disc  = (1 + r)**(-years)
        return pop['present_value_cost'] + annual * disc

class UtilityComponent:
    """Annual utility producer + PV QALY accumulation."""
    CONFIGURATION_DEFAULTS = {
        'utilities': {'mild':0.90, 'moderate':0.80, 'severe':0.60}
    }

    def setup(self, builder):
        builder.value.register_value_producer('annual_utility', self.compute)
        builder.value.register_value_modifier('present_value_qaly', self.accumulate)
        self.clock = builder.time.clock()

    def initialise_population(self, pop):
        pop['present_value_qaly'] = 0.0
        pop['age_at_entry']      = pop['age']

    def compute(self, pop, _):
        util_map = self.configuration.utilities
        return pop['severity'].map(util_map).values

    def accumulate(self, pop, annual, _):
        years = pop['age'] - pop['age_at_entry']
        r     = self.configuration.discount_rate
        disc  = (1 + r)**(-years)
        return pop['present_value_qaly'] + annual * disc

class TaxTransferComponent:
    """Compute Carer Payment via OpenFisca each step."""
    def setup(self, builder):
        self.of_system = CountryTaxBenefitSystem()
        self.builder   = SimulationBuilder(self.of_system)
        builder.value.register_value_producer('carer_payment', self.compute)

    def compute(self, pop, _):
        hh = self.builder.create_household()
        hh('household_income')[:] = pop['earned_income']
        self.builder.build()
        return hh.calculate('carer_payment')

# 4) Build a small synthetic population
# --------------------------------------------------
n = 5000
df = pd.DataFrame({
    'age':            np.random.randint(0,60, size=n),
    'severity':       np.random.choice(['mild','moderate','severe'], size=n, p=[0.5,0.3,0.2]),
    'earned_income':  np.random.normal(50_000, 10_000, size=n),
})
df['severity'] = df['severity'].astype('category').cat.set_categories(['mild','moderate','severe'])
pop = Population.from_data(df)

# 5) Run with InteractiveContext (Vivarium ≥0.3)
# --------------------------------------------------
sim = InteractiveContext(
    components = [CostComponent(), UtilityComponent(), TaxTransferComponent()],
    configuration = None,
    plugin_configuration = None,
    sim_name = 'IDMOD-lite-AUS'
)
sim.setup()
sim.initialize_simulants(pop)
sim.run()
results = sim.get_results()

# 6) Inspect results
# --------------------------------------------------
costs = results['present_value_cost']
qalys = results['present_value_qaly']
print(f"Mean PV cost  : A${costs.mean():,.0f}")
print(f"Mean PV QALYs : {qalys.mean():.2f}")


  Using cached openfisca_core-43.3.8-py3-none-any.whl.metadata (15 kB)
  Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl.metadata (10 kB)
Using cached sortedcontainers-2.4.0-py2.py3-none-any.whl (29 kB)
  Attempting uninstall: psutil
    Found existing installation: psutil 6.0.0
    Uninstalling psutil-6.0.0:
      Successfully uninstalled psutil-6.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [openfisca-core]m [openfisca-core]
  Cloning https://github.com/openfisca/openfisca-australia.git to /private/var/folders/m9/_g3wndrn04d80ys2382r700r0000gn/T/pip-req-build-0mbv58y6
  Running command git clone --filter=blob:none --quiet https://github.com/openfisca/openfisca-australia.git /private/var/folders/m9/_g3wndrn04d80ys2382r700r0000gn/T/pip-req-build-0mbv58y6
Username for 'https://github.com': 

In [7]:
!pip install vivarium vivarium-core vivarium-public-health openfisca-core numpy pandas scipy

In [4]:
import sys, numpy, pandas, scipy, tables
import vivarium, openfisca_core, openfisca_aotearoa
print("Python", sys.version.split()[0])
print("numpy", numpy.__version__)
print("pandas", pandas.__version__)
print("scipy", scipy.__version__)
print("tables", tables.__version__)
print("vivarium-core", vivarium.__version__)
print("openfisca-core", openfisca_core.__version__)
print("openfisca-aotearoa", openfisca_aotearoa.__version__)
EOF


Python 3.10.18
numpy 1.24.4
pandas 2.3.0
scipy 1.15.2
tables 3.10.1
vivarium-core 3.4.5


AttributeError: module 'openfisca_core' has no attribute '__version__'

In [5]:
# Install dependency (once; comment out afterwards)
# !pip install simpy numpy pandas

import simpy
import numpy as np
import pandas as pd

# --- PARAMETERS ---
DISCOUNT_RATE = 0.035
SEVERITY_CAP  = {'mild': 74, 'moderate': 68, 'severe': 59}
SEVERITY_P    = [0.5, 0.3, 0.2]

COST_COMMON = {
    'respite_care': 13024,
    'mental_health':30418,
    'hospital':      6800,
    'informal_care':25000,
}
COST_SEV = {
    'mild':     {'ndis_school': 26472,  'residential': 0},
    'moderate': {'ndis_school': 86472,  'residential': 0},
    'severe':   {'ndis_school':105972,  'residential':16973},
}
DSP_ANNUAL = 23254
UTIL       = {'mild':0.90, 'moderate':0.80, 'severe':0.60}

# --- SIMULATION LOGIC ---
def annual_cost(person, age):
    base = sum(COST_COMMON.values())
    sev_cfg = COST_SEV[person['severity']]
    add = sev_cfg['ndis_school'] + sev_cfg['residential']
    dsp = DSP_ANNUAL if age >= 20 else 0
    return base + add + dsp

def person_process(env, person, cost_acc, qaly_acc):
    age = person['age0']
    cap = SEVERITY_CAP[person['severity']]
    while age <= cap:
        disc = (1 + DISCOUNT_RATE)**-(age - person['age0'])
        c = annual_cost(person, age) * disc
        q = UTIL[person['severity']] * disc
        cost_acc[person['id']] += c
        qaly_acc[person['id']] += q
        age += 1
        yield env.timeout(1)

def run_sim(pop_size=5000, seed=2025):
    np.random.seed(seed)
    population = [{
        'id': i,
        'severity': np.random.choice(['mild','moderate','severe'], p=SEVERITY_P),
        'age0':     np.random.randint(0, 60)
    } for i in range(pop_size)]
    env = simpy.Environment()
    cost_acc = np.zeros(pop_size)
    qaly_acc = np.zeros(pop_size)
    for p in population:
        env.process(person_process(env, p, cost_acc, qaly_acc))
    # Fast-forward to year-cap (max 74 years of simulation)
    env.run(until=max(SEVERITY_CAP.values()) + 1)
    return cost_acc, qaly_acc

# --- EXECUTE & REPORT ---
costs, qalys = run_sim()
print(f"Mean PV cost  : A${costs.mean():,.0f}")
print(f"Mean PV QALYs : {qalys.mean():.2f}")


Mean PV cost  : A$3,235,632
Mean PV QALYs : 17.08


In [4]:
!pip install simpy

  Using cached simpy-4.1.1-py3-none-any.whl.metadata (6.1 kB)
Using cached simpy-4.1.1-py3-none-any.whl (27 kB)


In [6]:
import numpy as np
import pandas as pd

# --- PARAMETERS & CONFIGURATION ----------------------------------------------

# Simulation settings
POP_SIZE       = 5000     # Number of synthetic individuals
MC_DRAWS       = 1000     # Number of Monte Carlo counterfactual draws
START_YEAR     = 2025
DISCOUNT_RATE  = 0.035

# Severity categories & life-expectancy caps
SEVERITY_LEVELS = ['mild', 'moderate', 'severe']
SEVERITY_P      = [0.5, 0.3, 0.2]  # population mix
LIFE_CAPS       = {'mild': 74, 'moderate': 68, 'severe': 59}

# Annual cost components (2017 A$)
COST_COMMON = {
    'respite_care': 13024,
    'mental_health': 30418,
    'hospital':       6800,
    'informal_care': 25000,
}
COST_SEVERITY = {
    'mild':     {'ndis_school': 26472,  'residential': 0},
    'moderate': {'ndis_school': 86472,  'residential': 0},
    'severe':   {'ndis_school':105972, 'residential':16973},
}
DSP_COST = 23254  # Disability Support Pension
UTIL      = {'mild': 0.90, 'moderate': 0.80, 'severe': 0.60}

# Income distribution parameters for synthetic matching
INCOME_PARAMS = {
    'mild':     {'mu': np.log(50000), 'sigma': 0.5},
    'moderate': {'mu': np.log(40000), 'sigma': 0.6},
    'severe':   {'mu': np.log(30000), 'sigma': 0.7},
}

# --- SYNTHETIC COHORT GENERATION --------------------------------------------

np.random.seed(123)
cohort = pd.DataFrame({
    'id': np.arange(POP_SIZE),
    'sex': np.random.choice(['F','M'], POP_SIZE),
    'severity': np.random.choice(SEVERITY_LEVELS, size=POP_SIZE, p=SEVERITY_P),
    'age0': np.random.randint(0, 60, size=POP_SIZE)
})

# --- MICROSIMULATION FUNCTION ------------------------------------------------

def simulate_draw(cohort_df):
    """
    Perform one Monte Carlo draw:
    - Impute income by severity
    - Compute PV costs and QALYs per individual
    """
    # Copy cohort for this draw
    df = cohort_df.copy()
    
    # Synthetic income matching: log-normal by severity
    params = df['severity'].map(INCOME_PARAMS)
    df['income'] = np.exp(
        np.random.normal(params.apply(lambda x: x['mu']),
                         params.apply(lambda x: x['sigma']))
    )
    
    # Pre-allocate accumulators
    pv_costs = np.zeros(len(df))
    pv_qalys = np.zeros(len(df))
    
    # Compute PV cost and QALYs per individual
    for idx, person in df.iterrows():
        cap = LIFE_CAPS[person['severity']]
        ages = np.arange(person['age0'], cap + 1)
        # Discount factors
        disc = (1 + DISCOUNT_RATE) ** -(ages - person['age0'])
        
        # Annual cost vector
        common_cost = sum(COST_COMMON.values())
        sev_cfg = COST_SEVERITY[person['severity']]
        sev_cost = sev_cfg['ndis_school'] + sev_cfg['residential']
        dsp = np.where(ages >= 20, DSP_COST, 0)
        annual_costs = common_cost + sev_cost + dsp
        
        # Annual QALYs
        annual_qalys = UTIL[person['severity']]
        
        # Present-value totals
        pv_costs[idx] = np.sum(annual_costs * disc)
        pv_qalys[idx] = np.sum(annual_qalys * disc)
    
    return pv_costs, pv_qalys, df['income'].values

# --- RUN MONTE CARLO --------------------------------------------------------

all_costs = np.zeros((MC_DRAWS, POP_SIZE))
all_qalys = np.zeros((MC_DRAWS, POP_SIZE))
all_incomes = np.zeros((MC_DRAWS, POP_SIZE))

for draw in range(MC_DRAWS):
    c, q, inc = simulate_draw(cohort)
    all_costs[draw] = c
    all_qalys[draw] = q
    all_incomes[draw] = inc

# --- AGGREGATE RESULTS ------------------------------------------------------

# Mean and percentiles across draws
mean_cost  = all_costs.mean(axis=1).mean()
median_cost = np.median(all_costs.mean(axis=1))
p5_cost    = np.percentile(all_costs.mean(axis=1), 5)
p95_cost   = np.percentile(all_costs.mean(axis=1), 95)

mean_qaly  = all_qalys.mean(axis=1).mean()
median_qaly = np.median(all_qalys.mean(axis=1))
p5_qaly    = np.percentile(all_qalys.mean(axis=1), 5)
p95_qaly   = np.percentile(all_qalys.mean(axis=1), 95)

print("=== IDMOD-Lite Microsimulation Results ===")
print(f"Mean PV cost (mean of draws): A${mean_cost:,.0f}")
print(f"Median PV cost: A${median_cost:,.0f} (5th–95th: A${p5_cost:,.0f}–A${p95_cost:,.0f})\n")
print(f"Mean PV QALYs (mean of draws): {mean_qaly:.2f}")
print(f"Median PV QALYs: {median_qaly:.2f} (5th–95th: {p5_qaly:.2f}–{p95_qaly:.2f})")



=== IDMOD-Lite Microsimulation Results ===
Mean PV cost (mean of draws): A$3,233,715
Median PV cost: A$3,233,715 (5th–95th: A$3,233,715–A$3,233,715)

Mean PV QALYs (mean of draws): 17.02
Median PV QALYs: 17.02 (5th–95th: 17.02–17.02)


In [7]:
import numpy as np
import pandas as pd

# --- PARAMETERS & CONFIGURATION ----------------------------------------------

# Simulation settings
POP_SIZE      = 5000    # synthetic individuals
START_YEAR    = 2025
DISCOUNT_RATE = 0.035

# Monte Carlo draws for synthetic matching
MC_DRAWS = 1000

# Severity categories & life-expectancy caps (years)
SEVERITY_LEVELS = ['mild', 'moderate', 'severe']
SEVERITY_P      = [0.5, 0.3, 0.2]
LIFE_CAPS       = {'mild': 74, 'moderate': 68, 'severe': 59}

# Age-band distribution (national)
AGE_BANDS       = [(0,14), (15,64), (65,100)]
AGE_BAND_P      = [0.19, 0.65, 0.16]

# Education levels distribution
EDUCATION_LEVELS = ['primary', 'secondary', 'tertiary']
EDU_P            = [0.2, 0.5, 0.3]

# Annual cost components (2017 A$)
COST_COMMON = {
    'respite_care':   13024,
    'mental_health':  30418,
    'informal_care':  25000,
}
COST_SEV = {
    'mild':     {'ndis_school': 26472,  'residential': 0},
    'moderate': {'ndis_school': 86472,  'residential': 0},
    'severe':   {'ndis_school':105972,  'residential':16973},
}
DSP_COST  = 23254       # Disability Support Pension
HOSP_RATE = {'mild':0.5, 'moderate':1.5, 'severe':3.0}   # events per year
HOSP_COST = 8500        # cost per hospitalisation
UTIL      = {'mild':0.90, 'moderate':0.80, 'severe':0.60}

# Income distribution params: log-normal (mu, sigma) by severity & education
INCOME_PARAMS = {
    'mild':     {'primary':(10.5,0.5), 'secondary':(10.8,0.4), 'tertiary':(11.2,0.3)},
    'moderate': {'primary':(10.3,0.6), 'secondary':(10.6,0.5), 'tertiary':(10.9,0.4)},
    'severe':   {'primary':(10.1,0.7), 'secondary':(10.3,0.6), 'tertiary':(10.6,0.5)},
}

# --- SYNTHETIC COHORT GENERATION --------------------------------------------

np.random.seed(0)
cohort = pd.DataFrame({
    'id': np.arange(POP_SIZE),
    'sex': np.random.choice(['F','M'], POP_SIZE),
    'severity': np.random.choice(SEVERITY_LEVELS, POP_SIZE, p=SEVERITY_P),
    'age0': np.random.randint(0, 60, POP_SIZE),
    'education': np.random.choice(EDUCATION_LEVELS, POP_SIZE, p=EDU_P),
})
cohort['age_band'] = np.select(
    [cohort['age0'] <= 14, cohort['age0'].between(15,64), cohort['age0'] >=65],
    ['0-14','15-64','65+']
)

# Assign sampling weights to match age-band & severity distributions
band_map = {'0-14': AGE_BAND_P[0], '15-64': AGE_BAND_P[1], '65+': AGE_BAND_P[2]}
cohort['weight'] = cohort['age_band'].map(band_map) * cohort['severity'].map(dict(zip(SEVERITY_LEVELS, SEVERITY_P)))
cohort['weight'] *= POP_SIZE / cohort['weight'].sum()

# --- MICROSIMULATION --------------------------------------------------------

def simulate_individual(person, seed):
    """Simulate one individual through lifetime, returns PV cost & QALYs."""
    np.random.seed(seed + int(person.id))
    cap = LIFE_CAPS[person.severity]
    ages = np.arange(person.age0, cap+1)
    disc = (1 + DISCOUNT_RATE) ** -(ages - person.age0)
    
    # Income draw by MC draw
    mu, sigma = INCOME_PARAMS[person.severity][person.education]
    income = np.exp(np.random.normal(mu, sigma))
    
    # Annual cost vector
    common = sum(COST_COMMON.values())
    sev_cfg = COST_SEV[person.severity]
    sev_cost = sev_cfg['ndis_school'] + sev_cfg['residential']
    dsp = np.where(ages >= 20, DSP_COST, 0)
    hosp_events = np.random.poisson(HOSP_RATE[person.severity], size=len(ages))
    hosp_costs = hosp_events * HOSP_COST
    annual_costs = common + sev_cost + dsp + hosp_costs
    
    # Annual QALYs
    annual_qalys = UTIL[person.severity]
    
    pv_cost = np.sum(annual_costs * disc)
    pv_qaly = np.sum(annual_qalys * disc)
    return pv_cost, pv_qaly, income

# Run MC draws and aggregate
results = []
for draw in range(MC_DRAWS):
    costs = []
    qalys = []
    incomes = []
    for _, person in cohort.iterrows():
        c, q, inc = simulate_individual(person, seed=draw)
        costs.append(c)
        qalys.append(q)
        incomes.append(inc)
    # weighted means
    w = cohort['weight']
    results.append({
        'mean_cost': np.average(costs, weights=w),
        'mean_qaly': np.average(qalys, weights=w),
        'mean_income': np.average(incomes, weights=w),
    })

res_df = pd.DataFrame(results)

# --- OUTPUT SUMMARY ---------------------------------------------------------

print("IDMOD-Lite Microsimulation with enhanced features:")
print(f"Mean PV cost: {res_df['mean_cost'].mean():,.0f}",
      f"(5th–95th: {res_df['mean_cost'].quantile(0.05):,.0f}–{res_df['mean_cost'].quantile(0.95):,.0f})")
print(f"Mean PV QALYs: {res_df['mean_qaly'].mean():.2f}",
      f"(5th–95th: {res_df['mean_qaly'].quantile(0.05):.2f}–{res_df['mean_qaly'].quantile(0.95):.2f})")
print(f"Mean income: A${res_df['mean_income'].mean():,.0f}")


IDMOD-Lite Microsimulation with enhanced features:
Mean PV cost: 2,950,131 (5th–95th: 2,949,683–2,950,566)
Mean PV QALYs: 17.39 (5th–95th: 17.39–17.39)
Mean income: A$53,625


In [12]:
# 1) pip install numba joblib numpy pandas
import numpy as np
import pandas as pd
from numba import njit, prange
from joblib import Parallel, delayed

# --- PARAMETERS (as before) ---
POP_SIZE      = 5000
MC_DRAWS      = 1000
DISCOUNT_RATE = 0.035

SEVERITY_LEVELS = ['mild','moderate','severe']
SEVERITY_P      = np.array([0.5,0.3,0.2])
LIFE_CAPS       = np.array([74, 68, 59])
# Map severity -> index
sev_index = {s:i for i,s in enumerate(SEVERITY_LEVELS)}

# Cost arrays
common_cost = np.sum([13024, 30418, 25000])  # respite + mental + informal
ndis_school = np.array([26472, 86472,105972])
residential = np.array([0,0,16973])
dsp_annual  = 23254
hosp_rate   = np.array([0.5,1.5,3.0])
hosp_cost   = 8500
util        = np.array([0.90,0.80,0.60])

# --- JIT-compiled inner loop -----------------------------------------------
@njit
def simulate_one(id_, sev_idx, age0, seed):
    np.random.seed(seed + id_)
    cap = LIFE_CAPS[sev_idx]
    # Pre-allocate
    pv_cost = 0.0
    pv_qaly = 0.0
    # Year loop
    for age in range(age0, cap+1):
        disc = (1 + DISCOUNT_RATE) ** -(age - age0)
        # Costs
        base = common_cost
        sev_cost = ndis_school[sev_idx] + residential[sev_idx]
        dsp = dsp_annual if age >= 20 else 0
        hosp_events = np.random.poisson(hosp_rate[sev_idx])
        annual_cost = base + sev_cost + dsp + hosp_events * hosp_cost
        pv_cost += annual_cost * disc
        # QALYs
        pv_qaly += util[sev_idx] * disc
    return pv_cost, pv_qaly

# --- Prepare cohort arrays -----------------------------------------------
np.random.seed(0)
ids       = np.arange(POP_SIZE)
severities= np.random.choice(SEVERITY_LEVELS, POP_SIZE, p=SEVERITY_P)
sev_idxs  = np.array([sev_index[s] for s in severities])
age0s     = np.random.randint(0,60, size=POP_SIZE)

# --- Parallel Monte-Carlo -------------------------------------------------
def run_draw(draw_seed):
    costs = np.empty(POP_SIZE, dtype=np.float64)
    qalys = np.empty(POP_SIZE, dtype=np.float64)
    for i in range(POP_SIZE):
        c, q = simulate_one(i, sev_idxs[i], age0s[i], draw_seed)
        costs[i], xalys = c, q
    return costs.mean(), qalys.mean()

# Use all cores; verbose=5 shows progress
results = Parallel(n_jobs=-1, verbose=5)(
    delayed(run_draw)(draw) for draw in range(MC_DRAWS)
)
# Unpack
mean_costs = np.array([r[0] for r in results])
mean_qalys = np.array([r[1] for r in results])

# --- Summarize -------------------------------------------------------------
print("Mean PV cost :", f"A${mean_costs.mean():,.0f}", 
      f"(5–95%: A${np.percentile(mean_costs,5):,.0f}–A${np.percentile(mean_costs,95):,.0f})")
print("Mean PV QALYs:", f"{mean_qalys.mean():.2f}", 
      f"(5–95%: {np.percentile(mean_qalys,5):.2f}–{np.percentile(mean_qalys,95):.2f})")


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  54 tasks      | elapsed:    1.6s
/Users/doughnut/miniconda3/envs/idmod-lite/lib/python3.10/site-packages/numpy/core/_methods.py:181: RuntimeWarning: invalid value encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
[Parallel(n_jobs=-1)]: Done 586 tasks      | elapsed:    2.7s


Mean PV cost : A$3,282,587 (5–95%: A$3,282,058–A$3,283,079)
Mean PV QALYs: nan (5–95%: nan–nan)


[Parallel(n_jobs=-1)]: Done 981 out of 1000 | elapsed:    3.3s remaining:    0.1s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    3.3s finished


In [11]:
!pip install numba joblib

  Using cached joblib-1.5.1-py3-none-any.whl.metadata (5.6 kB)
Using cached joblib-1.5.1-py3-none-any.whl (307 kB)


In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

# --- CONFIG & PARAMETERS ----------------------------------------------------

POP_SIZE      = 5000
MC_DRAWS      = 1000
DISCOUNT_RATE = 0.035
START_YEAR    = 2025

SEVERITIES    = ['mild','moderate','severe']
SEV_P         = np.array([0.5,0.3,0.2])
LIFE_CAPS     = np.array([74,68,59])
SEV_MAP       = {s:i for i,s in enumerate(SEVERITIES)}

# Cost parameters
COMMON_COST  = np.sum([13024,30418,25000])  # respite+mental+informal
NDIS_SCHOOL  = np.array([26472,86472,105972])
RESIDENTIAL  = np.array([0,0,16973])
DSP_COST     = 23254
HOSP_RATE    = np.array([0.5,1.5,3.0])
HOSP_COST    = 8500
UTIL         = np.array([0.90,0.80,0.60])

# Scenario (WGS) parameters
WGS_COST            = 7557    # per genome
WGS_DIAG_YIELD_INC  = 0.10    # +10 pp
DIAG_ODYSSEY_COST   = 5000    # hypothetical per-person odyssey cost

# Benchmark for calibration
OBS_BENCHMARK = {'mild':200_000, 'moderate':500_000, 'severe':1_000_000}

# --- SYNTHETIC COHORT --------------------------------------------------------

np.random.seed(0)
cohort = pd.DataFrame({
    'id':       np.arange(POP_SIZE),
    'severity': np.random.choice(SEVERITIES, size=POP_SIZE, p=SEV_P),
    'age0':     np.random.randint(0,60,size=POP_SIZE),
    'sex':      np.random.choice(['F','M'], size=POP_SIZE),
})
cohort['sev_idx'] = cohort['severity'].map(SEV_MAP)

# --- CORE SIMULATION ---------------------------------------------------------

def simulate_one(sev, age0, seed, structural=False):
    np.random.seed(seed + int(age0*10+sev))
    if structural:
        # draw death age from uniform between age0 and cap
        cap = LIFE_CAPS[sev]
        death_age = np.random.randint(age0, cap+1)
        ages = np.arange(age0, death_age+1)
    else:
        cap = LIFE_CAPS[sev]
        ages = np.arange(age0, cap+1)
    disc = (1+DISCOUNT_RATE)**-(ages-age0)
    # cost lines
    base = COMMON_COST
    sev_cost = NDIS_SCHOOL[sev] + RESIDENTIAL[sev]
    dsp = np.where(ages>=20, DSP_COST, 0)
    hosp = np.random.poisson(HOSP_RATE[sev], size=ages.size)*HOSP_COST
    annual_cost = base + sev_cost + dsp + hosp
    pv_cost = np.sum(annual_cost * disc)
    # QALYs
    annual_qaly = UTIL[sev]
    pv_qaly = np.sum(annual_qaly * disc)
    return pv_cost, pv_qaly

# --- ANALYSIS FUNCTIONS ------------------------------------------------------

def run_base_cohort():
    # deterministic, no MC
    df = cohort.copy()
    df[['cost','qaly']] = df.apply(
        lambda r: simulate_one(r.sev_idx, r.age0, seed=0),
        axis=1, result_type='expand'
    )
    return df.groupby('severity').agg(
        mean_cost=('cost','mean'),
        mean_qaly=('qaly','mean'),
        count=('cost','size')
    )

def run_psa(n_jobs=-1):
    def draw(draw_seed):
        costs, qalys = [], []
        for _,r in cohort.iterrows():
            c,q = simulate_one(r.sev_idx, r.age0, draw_seed)
            costs.append(c); qalys.append(q)
        return np.mean(costs), np.mean(qalys)
    results = Parallel(n_jobs=n_jobs)(
        delayed(draw)(s) for s in range(MC_DRAWS)
    )
    psa_df = pd.DataFrame(results, columns=['cost','qaly'])
    return psa_df

def compute_ceac(psa_df, wtp_range=np.linspace(0,200000,50)):
    ceac = {}
    for wtp in wtp_range:
        nb = wtp * psa_df['qaly'] - psa_df['cost']
        ceac[wtp] = np.mean(nb > 0)
    return pd.Series(ceac)

def compute_evpi(psa_df, wtp):
    nb = wtp * psa_df['qaly'] - psa_df['cost']
    ep = np.mean(np.maximum(nb,0))
    enb = np.maximum(np.mean(nb),0)
    evpi = ep - enb
    return evpi

def one_way_sensitivity(param_name, lows, highs):
    # vary discount rate, caps, cost lines
    base_df = run_psa()
    base_cost = base_df['cost'].mean()
    sens = {}
    for low,high in zip(lows,highs):
        globals()[param_name] = low
        low_df = run_psa(n_jobs=1)
        globals()[param_name] = high
        high_df = run_psa(n_jobs=1)
        sens[low] = low_df['cost'].mean()
        sens[high] = high_df['cost'].mean()
    return pd.Series(sens)

def run_incremental_CEA():
    # standard vs WGS scenario: reduce odyssey cost and add genome cost
    def sim_scenario(seed, wgs=False):
        costs = []
        qalys = []
        for _,r in cohort.iterrows():
            c,q = simulate_one(r.sev_idx, r.age0, seed)
            if wgs and np.random.rand() < WGS_DIAG_YIELD_INC:
                c -= DIAG_ODYSSEY_COST
            if wgs:
                c += WGS_COST
                q += 0.01  # small QALY gain
            costs.append(c); qalys.append(q)
        return np.mean(costs), np.mean(qalys)
    # run MC
    std = np.array([sim_scenario(s, False) for s in range(MC_DRAWS)])
    wgs = np.array([sim_scenario(s, True)  for s in range(MC_DRAWS)])
    inc_cost = wgs[:,0] - std[:,0]
    inc_qaly = wgs[:,1] - std[:,1]
    icer     = np.median(inc_cost/inc_qaly)
    return inc_cost.mean(), inc_qaly.mean(), icer

def budget_impact(uptakes=[0.25,0.5,0.75], years=10):
    bi = {}
    pop = POP_SIZE
    base_cost,_ = run_incremental_CEA()[:2]
    for u in uptakes:
        bi[u] = [base_cost * pop * u for _ in range(years)]
    return pd.DataFrame(bi, index=range(1,years+1))

def distributional():
    df = run_base_cohort().reset_index().merge(cohort, on='severity')
    return df.groupby(['severity','sex']).agg({'cost':'mean','qaly':'mean'})

def calibration():
    df = run_base_cohort()
    errors = {}
    for sev,row in df.iterrows():
        obs = OBS_BENCHMARK[sev]
        sim = row.mean_cost
        errors[sev] = abs(sim - obs)/obs
    return pd.Series(errors)

def scenario_analyses(yields=[0.0,0.1,0.2], costs=[5000,7557,10000]):
    scenarios = []
    for y in yields:
        for c in costs:
            global WGS_DIAG_YIELD_INC, WGS_COST
            WGS_DIAG_YIELD_INC, WGS_COST = y, c
            mc, mq, icer = run_incremental_CEA()
            scenarios.append({'yield':y,'cost':c,'icer':icer})
    return pd.DataFrame(scenarios)

# Structural uncertainty: alternative survival
def run_structural():
    # use structural=True
    costs, qalys = [],[]
    for _,r in cohort.iterrows():
        c,q = simulate_one(r.sev_idx, r.age0, 0, structural=True)
        costs.append(c); qalys.append(q)
    return np.mean(costs), np.mean(qalys)

# --- EXAMPLE EXECUTION -------------------------------------------------------

# Base-case aggregates
base_agg = run_base_cohort()
# PSA
psa_df   = run_psa()
# CEAC
ceac     = compute_ceac(psa_df)
# EVPI @ WTP=50000
evpi_50k = compute_evpi(psa_df, 50000)
# One-way on discount_rate +/-50%
tornado  = one_way_sensitivity('DISCOUNT_RATE', [0.0,0.01], [0.05,0.1])
# Incremental CEA
inc_cost, inc_q, icer = run_incremental_CEA()
# Budget impact
bi_df    = budget_impact()
# Distributional
dist_df  = distributional()
# Calibration
cal_err  = calibration()
# Scenario analyses
scen_df  = scenario_analyses()
# Structural
str_cost, str_q = run_structural()

print("Base aggregates by severity:\n", base_agg)
print("\nCEAC:\n", ceac.head())
print("\nEVPI at 50k:", evpi_50k)
print("\nTornado sensitivity:\n", tornado)
print(f"\nIncremental CEA: Δcost={inc_cost:.0f}, ΔQALY={inc_q:.2f}, ICER={icer:.0f}")
print("\nBudget-impact (first 5 years):\n", bi_df.head())
print("\nDistributional by sev & sex:\n", dist_df)
print("\nCalibration errors:\n", cal_err)
print("\nScenario analyses:\n", scen_df)
print(f"\nStructural sim: cost={str_cost:.0f}, QALY={str_q:.2f}")


In [1]:

#!/usr/bin/env python3
import neworder as no
import pandas as pd
import numpy as np

# --- Configuration ---
START_YEAR = 2020
END_YEAR = 2100
STEP_YEAR = 1
# Total familial-ID cohort
TOTAL_POP = 25000000 * 0.015 * 0.165  # ~62k individuals

# --- Cost Parameters from Schofield et al. (2020) Table 1 ---
RESIDENTIAL_CARE_COST = 16973
RESIDENTIAL_DAILY_FEE = 18038
SDA_COST_RANGE = (4452, 59375)
SIL_WEEKLY_RANGE = (1727, 4951)
RESPITE_CARE_COST = 13024
TRANSFER_PAYMENTS = {
    "dsp_under20_independent": 13335,
    "dsp_under18_at_home": 8104,
    "dsp_18_20_at_home": 9393,
    "dsp_above20": 23254,
    "mobility_allowance_standard": 2493,
    "mobility_allowance_higher": 3487,
    "supported_employment": 5031
}
EDUCATION_COSTS = {
    ("primary","supplementary"):16107,
    ("primary","substantial"):27903,
    ("primary","extensive"):46733,
    ("secondary","supplementary"):18958,
    ("secondary","substantial"):30789,
    ("secondary","extensive"):49604,
    "early_intervention_per_hour":185
}

# --- Synthetic Population Generator ---
def generate_population(n):
    # Age 0-90
    ages = np.random.randint(0, 91, size=n)
    # Living arrangement categories
    living = np.random.choice(
        ["residential","sda","sil","home"],
        size=n,
        p=[0.2,0.1,0.3,0.4]
    )
    uses_respite = np.random.rand(n) < 0.3
    # School level by age
    school_level = [
        None if age<5 else ('primary' if age<12 else 'secondary')
        for age in ages
    ]
    support_level = np.random.choice(
        ['supplementary','substantial','extensive'],
        size=n, p=[0.5,0.3,0.2]
    )
    ei_hours = np.where(ages<5, np.random.randint(1, 11, size=n), 0)
    needs_mobility = np.random.rand(n) < 0.2
    employment_status = np.random.choice(
        ['none','supported','open'], size=n, p=[0.7,0.2,0.1]
    )
    return pd.DataFrame({
        'age': ages,
        'living_arrangement': living,
        'uses_respite': uses_respite,
        'school_level': school_level,
        'support_level': support_level,
        'ei_hours_per_week': ei_hours,
        'needs_mobility_aid': needs_mobility,
        'employment_status': employment_status
    })

# --- Population System ---
class IDMODPopulationSystem:
    def __init__(self, n=int(TOTAL_POP)):
        self.population = generate_population(n)
        self.population['annual_cost'] = 0.0

    def step(self):
        df = self.population
        df['annual_cost'] = 0.0
        # Residential care
        mask = df['living_arrangement']=='residential'
        df.loc[mask,'annual_cost'] += RESIDENTIAL_CARE_COST + RESIDENTIAL_DAILY_FEE
        # SDA
        mask = df['living_arrangement']=='sda'
        df.loc[mask,'annual_cost'] += np.random.uniform(*SDA_COST_RANGE, size=mask.sum())
        # SIL
        mask = df['living_arrangement']=='sil'
        df.loc[mask,'annual_cost'] += np.random.uniform(*SIL_WEEKLY_RANGE, size=mask.sum())*52
        # Respite
        mask = df['uses_respite']
        df.loc[mask,'annual_cost'] += RESPITE_CARE_COST
        # Transfer payments (credits)
        def dsp_cost(row):
            a, la = row['age'], row['living_arrangement']
            if a<18 and la=='home': return TRANSFER_PAYMENTS['dsp_under18_at_home']
            if a<20 and la=='independent': return TRANSFER_PAYMENTS['dsp_under20_independent']
            if 18<=a<20 and la=='home': return TRANSFER_PAYMENTS['dsp_18_20_at_home']
            if a>=20: return TRANSFER_PAYMENTS['dsp_above20']
            return 0
        df['annual_cost'] -= df.apply(dsp_cost,axis=1)
        # Mobility allowance
        df.loc[df['needs_mobility_aid'],'annual_cost'] -= TRANSFER_PAYMENTS['mobility_allowance_standard']
        # Supported employment
        df.loc[df['employment_status']=='supported','annual_cost'] -= TRANSFER_PAYMENTS['supported_employment']
        # Education
        def edu_cost(row):
            lvl = row['school_level']
            sup = row['support_level']
            if lvl in ['primary','secondary']:
                return EDUCATION_COSTS.get((lvl,sup),0)
            return row['ei_hours_per_week']*EDUCATION_COSTS['early_intervention_per_hour']*52
        df['annual_cost'] += df.apply(edu_cost, axis=1)

# --- Simulation Model ---
class MyTimeline(no.Timeline):
    def __init__(self):
        super().__init__()
    def at_start(self):
        return self.index() == START_YEAR
    def at_end(self):
        return self.index() >= END_YEAR

class IDMODSimulation(no.Model):
    def __init__(self):
        timeline = MyTimeline()
        super().__init__(timeline)
        self.systems = {'population': IDMODPopulationSystem()}

    def step(self):
        year = self.timeline.index()
        no.log(f"Year {year}")
        self.systems['population'].step()
        return not self.timeline.at_end()

    def finalise(self):
        pop = self.systems['population'].population
        no.log(f"Simulation complete. Mean annual cost: ${pop['annual_cost'].mean():.2f}")


<frozen importlib._bootstrap>:228: RuntimeWarning: mpi4py module not found, assuming serial mode
